# 06. Unified DB2 Comparative Study: Dictionary Learning vs. Molina (2025)
**Objective:** Head-to-head benchmarking of handcrafted features against Dictionary Learning on DB2 (2000 Hz).

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
import h5py
import numpy as np
import pandas as pd
import scipy.signal as signal
import pywt
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

sys.path.append(os.path.abspath('../'))
from src.config import PREPROCESSED_DIR
from src.features import extract_fft_magnitude, EMGDictionaryLearner, extract_time_domain_features

warnings.filterwarnings("ignore")

### 1. Feature Extraction Algorithms

In [2]:
def extract_molina_features_single_window(window: np.ndarray, fs: int = 2000) -> np.ndarray:
    """Extracts the 9-feature stack per channel as defined in Molina et al. (2025)."""
    feats = []
    n_samples, n_channels = window.shape
    
    for ch in range(n_channels):
        sig = window[:, ch]
        
        mav = np.mean(np.abs(sig))
        rms = np.sqrt(np.mean(sig ** 2))
        wl = np.sum(np.abs(np.diff(sig)))
        zc = np.sum(np.diff(np.sign(sig)) != 0)
        diff_2 = np.diff(sig, n=2)
        ssc = np.sum((diff_2[:-1] * diff_2[1:]) < 0)
        
        f, psd = signal.welch(sig, fs=fs, nperseg=n_samples)
        psd_norm = psd / (np.sum(psd) + 1e-12)
        spectral_entropy = -np.sum(psd_norm * np.log(psd_norm + 1e-12))
        mnf = np.sum(f * psd) / (np.sum(psd) + 1e-12)
        
        try:
            coeffs = pywt.wavedec(sig, 'db7', level=3, mode='symmetric')
            mdwt_energy = np.sum([np.sum(np.square(c)) for c in coeffs])
        except Exception:
            mdwt_energy = 0.0
            
        feats.extend([mav, rms, wl, zc, ssc, mnf, spectral_entropy, mdwt_energy])
        
    return np.array(feats)

def extract_molina_features(X_windows: np.ndarray, fs: int = 2000) -> np.ndarray:
    print(f"Extracting Molina Handcrafted Stack (8 features x {X_windows.shape[2]} channels)...")
    return np.array([extract_molina_features_single_window(w, fs=fs) for w in X_windows])

### 2. Load DB2 Dataset

In [3]:
data_path = os.path.join(PREPROCESSED_DIR, "DB2_S1_Windows.h5")
with h5py.File(data_path, 'r') as f:
    X_all = np.array(f['X'])
    y_all = np.array(f['y']).astype(np.int64)
    reps_all = np.array(f['reps'])

train_reps = [1, 3, 4, 6]
test_reps = [2, 5]

### 3. Experiment Execution

In [4]:
results = []

def evaluate_models(X_train, y_train, X_test, y_test, exp_name, feat_name):
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_train)
    X_te_sc = scaler.transform(X_test)
    
    rf = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)
    rf.fit(X_tr_sc, y_train)
    preds_rf = rf.predict(X_te_sc)
    
    svm = SVC(C=10.0, kernel='rbf', gamma='scale', random_state=42)
    svm.fit(X_tr_sc, y_train)
    preds_svm = svm.predict(X_te_sc)
    
    return {
        "Setup": exp_name,
        "Feature Method": feat_name,
        "Dimensions": X_train.shape[1],
        "RF Accuracy (%)": round(accuracy_score(y_test, preds_rf) * 100, 2),
        "RF Macro F1 (%)": round(f1_score(y_test, preds_rf, average='macro') * 100, 2),
        "SVM Accuracy (%)": round(accuracy_score(y_test, preds_svm) * 100, 2),
        "SVM Macro F1 (%)": round(f1_score(y_test, preds_svm, average='macro') * 100, 2),
    }

# ==========================================
# TEST A: 10 GESTURES (DB2 Exercise B, Native IDs 13-22)
# ==========================================
print("\n=== RUNNING 10-GESTURE EXPERIMENT ===")
mask_10g = np.isin(y_all, list(range(13, 23)))
X_10, y_10, reps_10 = X_all[mask_10g], y_all[mask_10g], reps_all[mask_10g]

tr_idx10 = np.isin(reps_10, train_reps)
te_idx10 = np.isin(reps_10, test_reps)
X_tr_10, y_tr_10, X_te_10, y_te_10 = X_10[tr_idx10], y_10[tr_idx10], X_10[te_idx10], y_10[te_idx10]

X_tr_10_molina = extract_molina_features(X_tr_10)
X_te_10_molina = extract_molina_features(X_te_10)
results.append(evaluate_models(X_tr_10_molina, y_tr_10, X_te_10_molina, y_te_10, "10 Gestures", "Molina Handcrafted"))

X_tr_10_freq = extract_fft_magnitude(X_tr_10)
X_te_10_freq = extract_fft_magnitude(X_te_10)
dl_10 = EMGDictionaryLearner(n_atoms=64, n_nonzero_coefs=5, random_state=42)
dl_10.fit(X_tr_10_freq)

X_tr_10_dl = np.hstack((dl_10.transform(X_tr_10_freq), extract_time_domain_features(X_tr_10)))
X_te_10_dl = np.hstack((dl_10.transform(X_te_10_freq), extract_time_domain_features(X_te_10)))
results.append(evaluate_models(X_tr_10_dl, y_tr_10, X_te_10_dl, y_te_10, "10 Gestures", "Hybrid DL (64 Atoms)"))

# ==========================================
# TEST B: ALL 49 GESTURES
# ==========================================
print("\n=== RUNNING 49-GESTURE EXPERIMENT ===")
mask_49g = y_all > 0
X_49, y_49, reps_49 = X_all[mask_49g], y_all[mask_49g], reps_all[mask_49g]

tr_idx49 = np.isin(reps_49, train_reps)
te_idx49 = np.isin(reps_49, test_reps)
X_tr_49, y_tr_49, X_te_49, y_te_49 = X_49[tr_idx49], y_49[tr_idx49], X_49[te_idx49], y_49[te_idx49]

X_tr_49_molina = extract_molina_features(X_tr_49)
X_te_49_molina = extract_molina_features(X_te_49)
results.append(evaluate_models(X_tr_49_molina, y_tr_49, X_te_49_molina, y_te_49, "49 Gestures", "Molina Handcrafted"))

X_tr_49_freq = extract_fft_magnitude(X_tr_49)
X_te_49_freq = extract_fft_magnitude(X_te_49)
dl_49 = EMGDictionaryLearner(n_atoms=128, n_nonzero_coefs=5, random_state=42)
dl_49.fit(X_tr_49_freq)

X_tr_49_dl = np.hstack((dl_49.transform(X_tr_49_freq), extract_time_domain_features(X_tr_49)))
X_te_49_dl = np.hstack((dl_49.transform(X_te_49_freq), extract_time_domain_features(X_te_49)))
results.append(evaluate_models(X_tr_49_dl, y_tr_49, X_te_49_dl, y_te_49, "49 Gestures", "Hybrid DL (128 Atoms)"))


=== RUNNING 10-GESTURE EXPERIMENT ===
Extracting Molina Handcrafted Stack (8 features x 12 channels)...
Extracting Molina Handcrafted Stack (8 features x 12 channels)...
Extracting FFT magnitudes...
Extracting FFT magnitudes...
Fitting Dictionary (64 atoms) on shape (2436, 3012)...
Dictionary learning complete.
Extracting Time-Domain features (MAV, RMS)...
Extracting Time-Domain features (MAV, RMS)...

=== RUNNING 49-GESTURE EXPERIMENT ===
Extracting Molina Handcrafted Stack (8 features x 12 channels)...
Extracting Molina Handcrafted Stack (8 features x 12 channels)...
Extracting FFT magnitudes...
Extracting FFT magnitudes...
Fitting Dictionary (128 atoms) on shape (16028, 3012)...
Dictionary learning complete.
Extracting Time-Domain features (MAV, RMS)...
Extracting Time-Domain features (MAV, RMS)...


### 4. Final Output

In [5]:
df_results = pd.DataFrame(results)
print("\n" + "=" * 105)
print(" DB2 BENCHMARK: PROPOSED DICTIONARY LEARNING vs. MOLINA HANDCRAFTED FEATURES")
print("=" * 105)
print(df_results.to_string(index=False))
print("=" * 105)


 DB2 BENCHMARK: PROPOSED DICTIONARY LEARNING vs. MOLINA HANDCRAFTED FEATURES
      Setup        Feature Method  Dimensions  RF Accuracy (%)  RF Macro F1 (%)  SVM Accuracy (%)  SVM Macro F1 (%)
10 Gestures    Molina Handcrafted          96            85.74            86.44             81.12             81.74
10 Gestures  Hybrid DL (64 Atoms)          88            83.95            84.86             78.53             79.14
49 Gestures    Molina Handcrafted          96            82.19            81.63             79.85             79.74
49 Gestures Hybrid DL (128 Atoms)         152            78.74            77.81             68.11             66.31
